In [1]:
import zipfile
import json
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

See [LSEG Terms](https://permid.org/terms) for the license associated with each of these fields.

Most of them are CC-BY 4.0, but the one we care most about, LEI, is CC0.

In [2]:
def jsonl_without_newlines(remaining):
    """
    The data *should* have been collected with newlines between each JSON object, but a few weren't.
    """
    try:
        data = json.loads(remaining)
    except json.JSONDecodeError as err:
        if err.msg == "Expecting value":
            return []
        elif err.msg == "Extra data":
            head = [json.loads(remaining[0:err.pos])]
            tail = jsonl_without_newlines(remaining[err.pos:])
            return head + tail
    else:
        return [data]

series_cik = []
series_lei = []
series_permid = []
series_name = []
series_website = []
series_regaddress = []
series_hqaddress = []
series_regphone = []
series_hqphone = []
series_domiciled_in = []
series_incorporated_in = []
series_is_public = []
series_exchange = []
series_ticker = []
with zipfile.ZipFile(os.path.expanduser("~/Box/dsi-core/11th-hour/idi-corporate-structure/CIK-to-LEI-from-refinitiv.zip")) as zf:
    for zipinfo in zf.infolist():
        if zipinfo.filename.startswith("OUTPUT-MDASS/CIK") and zipinfo.file_size != 0:
            cik = zipinfo.filename.split("/")[1].split(".")[0][3:]
            with zf.open(zipinfo.filename) as file:
                for data in jsonl_without_newlines(file.read()):
                    if data == {}:
                        continue
                    series_cik.append(cik)
                    series_lei.append(data.get("LEI", [pd.NA])[0])
                    series_permid.append(data["PERM ID"][0])
                    series_name.append(data["Organization Name"][0])
                    series_website.append(data.get("Website", [pd.NA])[0])
                    series_regaddress.append(data.get("Registered Address", [pd.NA])[0])
                    series_hqaddress.append(data.get("HQ Address", [pd.NA])[0])
                    series_regphone.append(data.get("Registered Phone", [pd.NA])[0])
                    series_hqphone.append(data.get("HQ Phone", [pd.NA])[0])
                    series_domiciled_in.append(data.get("Domiciled in", [pd.NA])[0])
                    series_incorporated_in.append(data.get("Incorporated in", [pd.NA])[0])
                    series_is_public.append(data["Public"][0] == "true")
                    if "mainQuoteId.mdaas" in data:
                        series_exchange.append(data["mainQuoteId.mdaas"][0]["Primary Exchange"][0])
                        series_ticker.append(data["mainQuoteId.mdaas"][0].get("Primary Ticker", [pd.NA])[0])
                    else:
                        series_exchange.append(pd.NA)
                        series_ticker.append(pd.NA)

lseg_df = pd.DataFrame({
    "cik": series_cik,
    "lei": series_lei,
    "permid": series_permid,
    "name": series_name,
    "website": series_website,
    "regaddress": series_regaddress,
    "hqaddress": series_hqaddress,
    "regphone": series_regphone,
    "hqphone": series_hqphone,
    "domiciled_in": series_domiciled_in,
    "incorporated_in": series_incorporated_in,
    "is_public": series_is_public,
    "exchange": series_exchange,
    "ticker": series_ticker,
})

In [3]:
lseg_df

,cik,lei,permid,name,website,regaddress,hqaddress,regphone,hqphone,domiciled_in,incorporated_in,is_public,exchange,ticker
0,0001866550,984500E705D15F2DF680,1-5076918515,Thoughtworks Holding Inc,https://investors.thoughtworks.com/,251 Little Falls Drive\nNew Castle County\nWIL...,"200 East Randolph Street, 25th Floor\nCHICAGO\...",13026365400,13123731000,United States,United States,False,NSM,TWKS
1,0001787414,<NA>,1-5071169340,Bogota Financial Corp,<NA>,7 St. Paul Street Suite 820\nBALTIMORE\nMARYLA...,819 Teaneck Road\nTEANECK\nNEW JERSEY\n07666\n...,<NA>,12018620660,United States,United States,True,NAS,BSBK
2,0002011169,<NA>,1-5087023447,GC Wealth Management RIA LLC,<NA>,DELAWARE\nUnited States\n,20 University Road\n4Th Floor\nCAMBRIDGE\nMASS...,<NA>,16284009980,United States,United States,False,<NA>,<NA>
3,0002008410,<NA>,1-5087013134,Berry Wealth Group LP,<NA>,TEXAS\nUnited States\n,920 Whitehead Dr.\nGRANBURY\nTEXAS\n76048\nUni...,<NA>,18175739595,United States,United States,False,<NA>,<NA>
4,0001689007,254900QZ1X0L69RAJQ17,1-5052165659,Ranger Global Real Estate Advisors LLC,https://rangerglobalre.com/,"108 W. 13th Street, Suite 100\nNew Castle Coun...",1801 Wewatta Street\n11th Fl.\nDENVER\nCOLORAD...,18885282677,17204770644,United States,United States,False,<NA>,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16751,0001091596,549300H0QI00R4LH8U41,1-4295900886,Nuo Therapeutics Inc,https://www.nuot.com/,"850 New Burton Road, Suite 201\nKent County\nD...","8285 El Rio, Suite 150\nHOUSTON\nTEXAS\n77054\...",18004831140,18322369060,United States,United States,True,PNK,AURX
16752,0001097218,549300IDZFNVQJYDHP15,1-5000063047,Nationwide Fund Advisors,https://www.nationwide.com/nationwide-fund-adv...,Corporation Trust Center\n1209 Orange Street\n...,10 West Nationwide Blvd\nCOLUMBUS\nOHIO\n43215...,13026587581,16102302800,United States,United States,False,<NA>,<NA>
16753,0000101778,1FRVQX2CRLGC1XLP5727,1-4295905140,Marathon Oil Corp,https://www.marathonoil.com/,251 Little Falls Drive\nNew Castle County\nWIL...,PO Box 3128\nHOUSTON\nTEXAS\n77253-3128\nUnite...,18009279800,17136296600,United States,United States,False,NYS,MRO
16754,0001909654,<NA>,1-5051004152,New Millennium Group LLC,<NA>,<NA>,10050 South State Street\nSANDY\nUTAH\n84070\n...,<NA>,18014469950,United States,United States,False,<NA>,<NA>


In [4]:
np.count_nonzero(lseg_df["lei"].notna()) / len(lseg_df)

0.4820362855096682

In [5]:
np.count_nonzero(lseg_df.query("is_public")["lei"].notna()) / len(lseg_df.query("is_public"))

0.6459483009003776

In [6]:
lseg_df[["cik", "lei", "permid", "name", "website", "regaddress", "hqaddress", "incorporated_in", "domiciled_in", "is_public"]].to_csv("cik-to-lei.csv", index=False)

In [7]:
ticker_df = pd.read_csv("~/Box/dsi-core/11th-hour/idi-corporate-structure/sec/ticker_to_cik.tsv", sep="\t", names=["ticker", "cik"], header=None, dtype=str)
ticker_df["cik"] = ticker_df["cik"].str.pad(10, fillchar="0")

In [8]:
tickers_matched = ticker_df.merge(lseg_df, on="cik")[["ticker_x", "ticker_y", "exchange"]]
tickers_matched

,ticker_x,ticker_y,exchange
0,msft,MSFT,NSM
1,brk-b,BRK.A,NYS
2,unh,UNH,NYS
3,jnj,JNJ,NYS
4,jnj,<NA>,<NA>
...,...,...,...
11842,hcicu,HCIC,NAS
11843,hcicw,HCIC,NAS
11844,hawlm,HAWLI,PNK
11845,hbanm,HBAN,NSM


In [9]:
investments_df = pd.read_csv("~/Box/dsi-core/11th-hour/idi-corporate-structure/sec/form13F_2024_Q4_shareholder_investments.tsv", sep="\t", encoding="latin-1")
investments_df["investor_cik"] = investments_df["investor_cik"].str[3:]

In [10]:
investments_matched = investments_df.merge(lseg_df, left_on="investor_cik", right_on="cik")

In [11]:
investments_matched.columns

Index(['stock_id', 'investor_cik', 'investor_name', 'investor_former_names',
       'investor_country', 'investor_region', 'other_investor_numbers',
       'other_investor_names', 'form_accession_number', 'form_report_date',
       'form_filing_date', 'stock_issuer', 'stock_cusip', 'stock_figi',
       'stock_ticker', 'stock_description', 'stock_value_x1000',
       'stock_shares_prn_amt', 'stock_prn_amt', 'stock_voting_auth_sole',
       'stock_voting_auth_shared', 'stock_voting_auth_none',
       'stock_exchange_codes', 'form_url', 'text', 'document', 'cik', 'lei',
       'permid', 'name', 'website', 'regaddress', 'hqaddress', 'regphone',
       'hqphone', 'domiciled_in', 'incorporated_in', 'is_public', 'exchange',
       'ticker'],
      dtype='object')

In [12]:
investments_matched[["investor_name", "investor_former_names", "name"]].drop_duplicates()

,investor_name,investor_former_names,name
0,WELLS FARGO & COMPANY/MN,"[""WELLS FARGO & CO/MN"", ""NORWEST CORP""]",Wells Fargo & Co
8,VANGUARD GROUP INC,"[""VANGUARD GROUP INC""]",Vanguard Group Inc
107,GOLDMAN SACHS GROUP INC,"[""GOLDMAN SACHS GROUP INC/""]",Goldman Sachs Group Inc
1286,NORTHERN TRUST CORP,[null],Northern Trust Corp
1471,TEACHERS RETIREMENT SYSTEM OF THE STATE OF KEN...,[null],Teachers Retirement System of State of Kentucky
...,...,...,...
2454145,NNS INVESTMENTS (CYPRUS) LTD,"[""DS INVESTMENTS LTD.""]",NNS Investments Cyprus Ltd
2480818,RADCLIFF MANAGEMENT LLC,[null],Radcliff Management LLC
2482069,PARADIGM OPERATIONS LP,[null],Paradigm Operations LP
2486994,POWER CORP OF CANADA,"[""POWER CORP OF CANADA ...",Power Corporation of Canada


In [13]:
shareholder_df = pd.read_csv("~/Box/dsi-core/11th-hour/idi-corporate-structure/sec/form13F_2024_Q4_shareholder_metadata.tsv", sep="\t")
shareholder_df["cik"] = shareholder_df["cik"].str[3:]

In [14]:
shareholders_matched = shareholder_df.merge(lseg_df, on="cik")

In [15]:
shareholders_matched.columns

Index(['id', 'cik', 'name_x', 'former_names', 'street', 'secondary', 'city',
       'state_or_country', 'zip_code', 'lei', 'permid', 'name_y', 'website',
       'regaddress', 'hqaddress', 'regphone', 'hqphone', 'domiciled_in',
       'incorporated_in', 'is_public', 'exchange', 'ticker'],
      dtype='object')

In [16]:
shareholders_matched[["name_x", "name_y", "street", "secondary", "city", "state_or_country", "zip_code", "regaddress", "hqaddress"]]

,name_x,name_y,street,secondary,city,state_or_country,zip_code,regaddress,hqaddress
0,CNA FINANCIAL CORP,CNA Financial Corp,CNA,151 N. FRANKLIN,CHICAGO,IL,60606,Corporation Trust Center\n1209 Orange Street\n...,151 N Franklin St\nCHICAGO\nILLINOIS\n60606-18...
1,SYNOVUS FINANCIAL CORP,Synovus Financial Corp,33 W 14TH STREET,NaN,COLUMBUS,GA,31901,"1111 Bay Avenue, Suite 350\nCOLUMBUS\nGEORGIA\...",33 W 14Th Street\nCOLUMBUS\nGEORGIA\n31901\nUn...
2,CENTRAL SECURITIES CORP,Central Securities Corp,630 FIFTH AVENUE,SUITE 820,NEW YORK,NY,10111,Corporation Trust Center\n1209 Orange Street\n...,630 Fifth Avenue\nSuite 820\nNEW YORK\nNEW YOR...
3,CHASE INVESTMENT COUNSEL CORP,Chase Investment Counsel Corp,"350 OLD IVY WAY, SUITE 100",NaN,CHARLOTTESVILLE,VA,22903-4897,350 Old Ivy Way Ste 100\nCHARLOTTESVILLE\nVIRG...,"350 Old Ivy Way, Suite 100\nCHARLOTTESVILLE\nV..."
4,"Virtus Investment Advisers, LLC",Virtus Investment Advisers LLC,ONE FINANCIAL PLAZA,26TH FLOOR,HARTFORD,CT,06103,84 State Street\nBOSTON\nMASSACHUSETTS\n02109\...,One Financial Plaza\n26Th Floor\nHARTFORD\nCON...
...,...,...,...,...,...,...,...,...,...
7948,CIBRA Capital Ltd,Cibra Capital Ltd,12 HAMMERSMITH GROVE,CIBRA CAPITAL LTD,LONDON,X0,W6 7AP,<NA>,12 Hammersmith Grove\nCibra Capital Ltd\nW6 7A...
7949,Chancellor Financial Group WB LP,Chancellor Financial Group WB LP,60 PUBLIC SQUARE,SUITE 600,WILKES-BARRE,PA,18701,15 Public Sq Ste 400\nWILKES BARRE\nPENNSYLVAN...,60 Public Square\nSuite 600\nWILKES-BARRE\nPEN...
7950,"DSG Capital Advisors, LLC",DSG Capital Advisors LLC,7760 FRANCE AVENUE SOUTH,SUITE 815,EDINA,MN,55435,SOUTH DAKOTA\nUnited States\n,7760 France Avenue South\nSuite 815\nEDINA\nMI...
7951,M1 Capital Management LLC,M1 Capital Management LLC,7 W SQUARE LAKE ROAD,SUITE 112,BLOOMFIELD HILLS,MI,48302,"28175 Haggerty Rd, Suite 170\nNOVI\nMICHIGAN\n...",7 W Square Lake Road\nSuite 112\nBLOOMFIELD HI...


In [17]:
gleif_df = pd.read_csv(
    "~/Box/dsi-core/11th-hour/idi-corporate-structure/gleif/20250807-0800-gleif-goldencopy-lei2-golden-copy.csv",
    usecols=[
        "LEI",
        "Entity.LegalName",
        "Entity.LegalAddress.FirstAddressLine",
        "Entity.LegalAddress.AdditionalAddressLine.1",
        "Entity.LegalAddress.AdditionalAddressLine.2",
        "Entity.LegalAddress.AdditionalAddressLine.3",
        "Entity.LegalAddress.City",
        "Entity.LegalAddress.Region",
        "Entity.LegalAddress.Country",
        "Entity.LegalAddress.PostalCode",
        "Entity.HeadquartersAddress.FirstAddressLine",
        "Entity.HeadquartersAddress.AdditionalAddressLine.1",
        "Entity.HeadquartersAddress.AdditionalAddressLine.2",
        "Entity.HeadquartersAddress.AdditionalAddressLine.3",
        "Entity.HeadquartersAddress.City",
        "Entity.HeadquartersAddress.Region",
        "Entity.HeadquartersAddress.Country",
        "Entity.HeadquartersAddress.PostalCode",
    ],
    dtype=str,
)

In [18]:
gleif_merged = gleif_df.merge(lseg_df, left_on="LEI", right_on="lei")

In [19]:
gleif_merged.columns

Index(['LEI', 'Entity.LegalName', 'Entity.LegalAddress.FirstAddressLine',
       'Entity.LegalAddress.AdditionalAddressLine.1',
       'Entity.LegalAddress.AdditionalAddressLine.2',
       'Entity.LegalAddress.AdditionalAddressLine.3',
       'Entity.LegalAddress.City', 'Entity.LegalAddress.Region',
       'Entity.LegalAddress.Country', 'Entity.LegalAddress.PostalCode',
       'Entity.HeadquartersAddress.FirstAddressLine',
       'Entity.HeadquartersAddress.AdditionalAddressLine.1',
       'Entity.HeadquartersAddress.AdditionalAddressLine.2',
       'Entity.HeadquartersAddress.AdditionalAddressLine.3',
       'Entity.HeadquartersAddress.City', 'Entity.HeadquartersAddress.Region',
       'Entity.HeadquartersAddress.Country',
       'Entity.HeadquartersAddress.PostalCode', 'cik', 'lei', 'permid', 'name',
       'website', 'regaddress', 'hqaddress', 'regphone', 'hqphone',
       'domiciled_in', 'incorporated_in', 'is_public', 'exchange', 'ticker'],
      dtype='object')

In [20]:
gleif_merged[[
    "Entity.LegalName", "name",
    "Entity.LegalAddress.FirstAddressLine", "Entity.LegalAddress.City", "Entity.LegalAddress.Region", "Entity.LegalAddress.Country", "Entity.LegalAddress.PostalCode",
    "regaddress",
]]

,Entity.LegalName,name,Entity.LegalAddress.FirstAddressLine,Entity.LegalAddress.City,Entity.LegalAddress.Region,Entity.LegalAddress.Country,Entity.LegalAddress.PostalCode,regaddress
0,"ARISTEIA CAPITAL, L.L.C.",Aristeia Capital LLC,C/o The Corporation Trust Company,Wilmington,US-DE,US,19801,Corporation Trust Center\n1209 Orange Street\n...
1,WESTINGHOUSE AIR BRAKE TECHNOLOGIES CORPORATION,Westinghouse Air Brake Technologies Corp,C/O CORPORATION SERVICE COMPANY,WILMINGTON,US-DE,US,19808,251 Little Falls Dr\nWILMINGTON\nDELAWARE\n198...
2,NUVEEN MORTGAGE AND INCOME FUND,Nuveen Mortgage and Income Fund,C/O C T CORPORATION SYSTEM,BOSTON,US-MA,US,02110,"155 Federal St., Suite 700\nBOSTON\nMASSACHUSE..."
3,PACIFIC LIFE FUND ADVISORS LLC,Pacific Life Fund Advisors LLC,C/O CORPORATION SERVICE COMPANY,WILMINGTON,US-DE,US,19808,251 Little Falls Drive\nNew Castle County\nWIL...
4,Weyerhaeuser Company,Weyerhaeuser Co,C/O Kristy T Harlan,Seattle,US-WA,US,98104,300 Deschutes Way Sw Ste 208 Mc-Csc1\nTUMWATER...
...,...,...,...,...,...,...,...,...
8072,PIMCO MUNICIPAL INCOME FUND II,PIMCO Municipal Income Fund II,C/O CORPORATION SERVICE COMPANY,BOSTON,US-MA,US,02109,84 State St\nBOSTON\nMASSACHUSETTS\n02109\nUni...
8073,ATI INC.,ATI Inc,C/O THE CORPORATION TRUST COMPANY,WILMINGTON,US-DE,US,19801,Corporation Trust Center\n1209 Orange Street\n...
8074,"RYDER SYSTEM, INC.",Ryder System Inc,C/O Corporate Creations Network Inc.,North Palm Beach,US-FL,US,33408-3811,801 North US Highway 1\nNORTH PALM BEACH\nFLOR...
8075,BANCO BTG PACTUAL S.A.,Banco BTG Pactual SA,PR BOTAFOGO,RIO DE JANEIRO,BR-RJ,BR,22250-911,"Pr Botafogo, Bloco II Salao 501, Sala 601\nBot..."


In [21]:
gleif_merged[[
    "Entity.LegalName", "name",
    "Entity.HeadquartersAddress.FirstAddressLine", "Entity.HeadquartersAddress.City", "Entity.HeadquartersAddress.Region", "Entity.HeadquartersAddress.Country", "Entity.HeadquartersAddress.PostalCode",
    "hqaddress",
]]

,Entity.LegalName,name,Entity.HeadquartersAddress.FirstAddressLine,Entity.HeadquartersAddress.City,Entity.HeadquartersAddress.Region,Entity.HeadquartersAddress.Country,Entity.HeadquartersAddress.PostalCode,hqaddress
0,"ARISTEIA CAPITAL, L.L.C.",Aristeia Capital LLC,3rd Floor,Greenwich,US-CT,US,06830,One Greenwich Plaza\nSuite 300\nGREENWICH\nCON...
1,WESTINGHOUSE AIR BRAKE TECHNOLOGIES CORPORATION,Westinghouse Air Brake Technologies Corp,C/O CORPORATION SERVICE COMPANY,WILMINGTON,US-DE,US,19808,30 Isabella Street\nPITTSBURGH\nPENNSYLVANIA\n...
2,NUVEEN MORTGAGE AND INCOME FUND,Nuveen Mortgage and Income Fund,"C/O NUVEEN FUND ADVISORS, LLC",CHICAGO,US-IL,US,60606,333 W. Wacker Dr.\nCHICAGO\nILLINOIS\n60606\nU...
3,PACIFIC LIFE FUND ADVISORS LLC,Pacific Life Fund Advisors LLC,700 Newport Center Drive,Newport Beach,US-CA,US,92660,700 Newport Center Dr\nNEWPORT BEACH\nCALIFORN...
4,Weyerhaeuser Company,Weyerhaeuser Co,C/O Devin Stockfish,Federal Way,US-WA,US,98003,220 Occidental Avenue South\nSEATTLE\nWASHINGT...
...,...,...,...,...,...,...,...,...
8072,PIMCO MUNICIPAL INCOME FUND II,PIMCO Municipal Income Fund II,C/O Pacific Investment Management Company LLC,Newport Beach,US-CA,US,92660,1633 Broadway\nNEW YORK\nNEW YORK\n10019\nUnit...
8073,ATI INC.,ATI Inc,1000 6 PPG Place,Pittsburgh,US-PA,US,15222,2021 McKinney Avenue\nSuite 1100\nDALLAS\nTEXA...
8074,"RYDER SYSTEM, INC.",Ryder System Inc,11690 NW 105 ST,MIAMI,US-FL,US,33178-1103,2333 Ponce De Leon # 700\nMIAMI\nFLORIDA\n3313...
8075,BANCO BTG PACTUAL S.A.,Banco BTG Pactual SA,Andar 5 6 7,Rio de Janeiro,BR-RJ,BR,22250-040,"Praia de Botafogo, 501, 5 e 6 andares\nBotafog..."


In [22]:
gleif_merged[["Entity.LegalAddress.Country", "Entity.HeadquartersAddress.Country", "incorporated_in", "domiciled_in"]].drop_duplicates()

,Entity.LegalAddress.Country,Entity.HeadquartersAddress.Country,incorporated_in,domiciled_in
0,US,US,United States,United States
12,CA,CA,Canada,Canada
15,GB,GB,United Kingdom,United Kingdom
17,DE,DE,Germany,Germany
30,SE,SE,Sweden,Sweden
...,...,...,...,...
7342,KY,NL,Cayman Islands,Netherlands
7364,US,US,United States,Sweden
7484,PA,PA,Panama,United States
7645,LR,US,Liberia,United States
